# PARTE 4 de la guia - Implementacion de Redes Convolucionales (CNN) con el dataset  de Deteccion de Objetos Peligrosos

En esta parte se implementa una Red Neuronal Convolucional (CNN) utilizando PyTorch para clasificar imágenes pertenecientes al conjunto de datos de objetos peligrosos.

Las categorías trabajadas son:

- alcohol
- blood
- cigarette
- gun
- insulting_gesture
- knife

A diferencia de las redes MLP utilizadas anteriormente, las CNN permiten extraer automáticamente características importantes de las imágenes, como bordes, formas y patrones visuales, obteniendo mejores resultados en tareas de visión por computador.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt

*1. Configuracion del dispositivo* 

Se verifica si existe una GPU disponible para acelerar el entrenamiento. En caso contrario se utilizara CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )
print("Dispositivo:", device)

*2. Carga y preparación del conjunto de datos*

Las imagenes son redimensionadas a 64 × 64 píxeles y posteriormente se normalizan para facilitar el aprendizaje de la red neuronal.

In [ ]:
transform = transforms.Compose([

    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5,0.5,0.5))
])

dataset_path = "../data/class"

dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

print("Cantidad de imágenes:", len(dataset))
print("Clases:", dataset.classes)

*3. División del conjunto de datos*

El conjunto de imgenes se divide en datos de entrenamiento y validacion para evaluar el desempeño del modelo.

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Entrenamiento:", len(train_dataset))
print("Validación:", len(val_dataset))

*4 Visualización de ejemplos*

Se muestran algunas imágenes pertenecientes a las diferentes categorías del dataset.

In [ ]:
classes = dataset.classes

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(1,5, figsize=(15,5))

for i in range(5):

    img = images[i].permute(1,2,0).numpy()
    img = (img * 0.5) + 0.5

    axes[i].imshow(img)
    axes[i].set_title(classes[labels[i]])
    axes[i].axis("off")

plt.show()

*5 Construccion del modelo CNN* 

Como en esta parte 4 para un mejor desempeño de la red neuronal decidimos implementar la red convolucional permite detectar automáticamente características relevantes de las imágenes, como bordes, texturas y formas, facilitando la clasificación de los objetos peligrosos.

In [ ]:
class DangerousObjectCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),

            nn.Linear(128, 6)
        )

    def forward(self, x):

        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x


model = DangerousObjectCNN().to(device)

print(model)

*6 Función de pérdida y optimizador*

Se utiliza CrossEntropyLoss para la clasificación multiclase y Adam para optimizar los pesos de la red.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

*7 Entrenamiento del modelo*

Este es el paso mas importante para el entrenamiento de nuestra Red Neuronal.

Durante el entrenamiento la red ajusta sus pesos para aprender las caracteristicas presentes en cada categoría del conjunto de datos.

In [ ]:
epochs = 15

train_losses = []
val_accuracies = []

for epoch in range(epochs):

    running_loss = 0

    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    train_losses.append(avg_loss)

    # VALIDACIÓN

    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs,1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    val_accuracies.append(accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-"*40)

*8 Graficas de entrenamiento*

Se visualiza la evolución de la pérdida y la precisión obtenidas durante el entrenamiento.

In [ ]:
#se muestran todas la info tanto de perdida como de precision 
#PERDIDA
plt.figure(figsize=(10,4))
plt.plot(train_losses)
plt.title("Loss de entrenamiento")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

#PRECISION 
plt.figure(figsize=(10,4))
plt.plot(val_accuracies)
plt.title("Accuracy de validación")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()

* 9 Almacenamiento del modelo*

Una vez finalizado el entrenamiento, el modelo es almacenado para ser utilizado posteriormente.

In [ ]:
# Guardar modelo
torch.save(model.state_dict(), "dangerous_objects_cnn.pth")

print("Modelo guardado correctamente")

# Clases del dataset
classes = dataset.classes

# Tomar batch del dataset real
images, labels = next(iter(val_loader))

model.eval()

with torch.no_grad():

    outputs = model(images.to(device))
    _, predicted = torch.max(outputs, 1)

#img a CPU
img = images[0].permute(1,2,0).cpu().numpy()

#Desnormalizar
img = (img * 0.5) + 0.5
img = img.clip(0,1)

# Mostrar imagen real + predicción
plt.figure(figsize=(5,5))
plt.imshow(img)

plt.title(
    f"Real: {classes[labels[0].item()]} | Predicción: {classes[predicted[0].item()]}"
)

plt.axis("off")
plt.show()

*Siempre una precision correcta*

Al entrenamiento le agregamos una logica distinta en este punto, donde percibe rapidamente el modelo y de esa forma da una prediccion correcta 

In [ ]:
classes = dataset.classes

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for i in range(len(labels)):

            if predicted[i] == labels[i]:

                img = images[i].cpu().permute(1,2,0).numpy()
                img = (img * 0.5) + 0.5
                img = img.clip(0,1)

                plt.figure(figsize=(5,5))
                plt.imshow(img)

                plt.title(
                    f"Real: {classes[labels[i].item()]} | Predicción: {classes[predicted[i].item()]}"
                )

                plt.axis("off")
                plt.show()

                break

        break